In [ ]:
from pathlib import Path
current_dir = Path().resolve()
from microlive.imports import *
from microlive import microscopy as mi
main_dir = Path().resolve().parent
import tasep_models as tm
from tasep_models import *
from Bio import pairwise2
from Bio.pairwise2 import format_alignment  
import multiprocessing as mp
import os
mp.set_start_method('spawn', force=True)
os.environ['PYTHONWARNINGS'] = 'ignore::UserWarning:multiprocessing.resource_tracker'

In [ ]:
# reload module microlive
import importlib
importlib.reload(mi)

In [ ]:
current_dir = Path().resolve()
print(f"Current directory: {current_dir}")

In [ ]:
# Load the sequences.
HA_TAG = 'YPYDVPDYA'
GFP_TAG = 'LEFVTAA'  
MCHERRY_TAG = 'QYERAEG'
TAG_list = [HA_TAG, GFP_TAG, MCHERRY_TAG]

In [ ]:
GFPuv = 'SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYAVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYITADKQKNGIKANFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSKLSKDPNEKRDHMVLLEFVTAAGITHGMDELYK'
mCherry ='GVSKGEEDNMAIIKEFMRFKVHMEGSVNGHEFEIEGEGEGRPYEGTQTAKLKVTKGGPLPFAWDILSPQFMYASKAYVKHPADIPDYLKLSFPEGFKWERVMNFEDGGVVTVTQDSSLQDGEFIYKVKLRGTNFPSDGPVMQKKTMGWEASSERMYPEDGALKGEIKQRLKLKDGGHYDAEVKTTYKAKKPVQLPGAYNVNIKLDITSHNEDYTIVEQYERAEGRHSTGGMDELYKG'

In [ ]:
# Load sequences.
gene_sequence_4xmGFPuvq_2xmCherryq = current_dir.parents[1].joinpath('data', 'gene_sequences', 'pRS032 (pUB-RBsmHA-4xmGFPuvq-2xmCherryq-24xMS2).dna') # slow folding
gene_sequence_4xsfGFPq_2xmCherryq = current_dir.parents[1].joinpath('data', 'gene_sequences', 'pRS027 (pUB-RBsmHA-4xsfGFPq-2xmCherryq-24xMS2).dna') # fast folding
gene_sequence_1xsfGFPq_5xmCherryq = current_dir.parents[1].joinpath('data', 'gene_sequences', 'pRS031 (pUB-RBsmHA-1xsfGFPq-5xmCherryq-24xMS2).dna')

gene_sequence_pNZ251 = current_dir.parents[1].joinpath('data', 'gene_sequences', 'pNZ251 (pUB-mRuby2HA-1xsfGFP-24xMS2).dna')


In [ ]:
def read_gene_sequence(file_path, TAG_list):
    protein, rna, _, indexes_tags, _, seq_record, graphic_features = read_sequence(
        seq=file_path, TAG=TAG_list, min_protein_length=50
    )
    plasmid_figure = plot_plasmid(seq_record, graphic_features, figure_width=25, figure_height=3)
    gene_length = len(protein) + 1  # adding 1 to account for the stop codon
    probe_data = {}
    for i, (tag, tag_positions) in enumerate(zip(TAG_list, indexes_tags)):
        probe_vector = create_probe_vector(tag_positions, gene_length)
        probe_data[f'tag_{i}'] = {
            'tag_sequence': tag,
            'positions': tag_positions,
            'position_cumulative_vector': probe_vector
        }
    return {
        "protein": protein,
        "rna": rna,
        "gene_length": gene_length,
        "probe_data": probe_data,  # Dictionary with all probe information
        "plasmid_figure": plasmid_figure,
        "seq_record": seq_record,
        "graphic_features": graphic_features,
        "num_probes": len(indexes_tags)
    }
#data_sequence = read_gene_sequence(gene_sequence_1xsfGFPq_5xmCherryq, TAG_list)
data_sequence = read_gene_sequence(gene_sequence_4xsfGFPq_2xmCherryq, TAG_list)
#data_sequence = read_gene_sequence(gene_sequence_pNZ251, [HA_TAG, GFP_TAG])





In [ ]:
# # Initial conditions
gene_length = data_sequence['gene_length']
exp_decorrelation = 600
G0 = 0.04
ki = 1/(G0*exp_decorrelation)#0.005  # Initiation rate
global_elongation_rate = gene_length / exp_decorrelation
print('elongation rate:', global_elongation_rate )
print('initiation rate:', ki    )


In [ ]:
# print the postion of each tag
tag_0_positions = data_sequence['probe_data']['tag_0']['positions']  # HA_TAG (earlier)
tag_1_positions = data_sequence['probe_data']['tag_1']['positions']  # GFP_TAG (later)
print(f"Tag 0 (HA) positions: {tag_0_positions}")
print(f"Tag 1 (GFP) positions: {tag_1_positions}")

In [ ]:


#global_elongation_rate = 5  # Elongation rates for positions 1 to N-1
number_repetitions = 200
burnin_time = 2500
t_max = 361*5 #timePerturbationApplication + 25*60  # Maximum time
step_size_in_sec = 5 # 5
time_array = np.arange(0, t_max, step_size_in_sec)
#number_tested_parameters = 5
MAD_THRESHOLD_FACTOR = 4
folding_delay = 60*2 # seconds
evaluatingFRAP = False
timePerturbationApplication = None # 361*5 # seconds
ke = calculate_codon_elongation_rates (data_sequence['rna'], global_elongation_rate=global_elongation_rate)

intensity_vector_first_signal_ode,intensity_vector_second_signal_ode = simulate_TASEP_ODE(ki, ke, 
                                                                                          gene_length = data_sequence['gene_length'], 
                                                                                          t_max = t_max,
                                                                                          first_probe_position_vector = data_sequence['probe_data']['tag_0']['position_cumulative_vector'],
                                                                                          second_probe_position_vector = data_sequence['probe_data']['tag_1']['position_cumulative_vector'],
                                                                                          burnin_time = burnin_time,
                                                                                          time_interval_in_seconds = step_size_in_sec)


constant_elongation_rate = None # this is a signal to the SSA to use the elongation rates considering sequence variability.
list_ribosome_trajectories,list_occupancy_output, matrix_intensity_first_signal_RT, matrix_intensity_second_signal_RT = simulate_TASEP_SSA(ki, ke, 
                                                                                                                        gene_length = data_sequence['gene_length'],
                                                                                                                        t_max = t_max,
                                                                                                                        time_interval_in_seconds = step_size_in_sec,
                                                                                                                        number_repetitions = number_repetitions, 
                                                                                                                        first_probe_position_vector = data_sequence['probe_data']['tag_0']['position_cumulative_vector'],
                                                                                                                        second_probe_position_vector = data_sequence['probe_data']['tag_1']['position_cumulative_vector'],
                                                                                                                        burnin_time = burnin_time,
                                                                                                                        folding_delay=folding_delay,
                                                                                                                        constant_elongation_rate = constant_elongation_rate,
                                                                                                                        fast_output = False,
                                                                                                                        evaluatingFRAP = evaluatingFRAP,
                                                                                                                        timePerturbationApplication =timePerturbationApplication,
                                                                                                                        gate_by_first_signal_per_event=True)


In [ ]:
#plot_trajectories(matrix_intensity_first_signal_RT, intensity_vector_first_signal_ode, time_array, number_repetitions,plot_color='forestgreen')
#plot_trajectories(matrix_intensity_second_signal_RT, intensity_vector_second_signal_ode, time_array, number_repetitions,plot_color='indigo')

In [ ]:
plot_dual_signal_trajectories(matrix_intensity_first_signal_RT, matrix_intensity_second_signal_RT, 
                              time_array, trajectory_index=1, 
                              colors=['forestgreen', 'indigo'], 
                              labels=['HA_TAG', 'GFP_TAG'], 
                              normalize=True)

In [ ]:
# def plot_RibosomeMovement_and_Microscope(RibosomePositions, IntensityVector, probePositions, SecondIntensityVector=None, second_probePositions=None, fileNameGif='temp_gif', color='red', second_color='lime', FrameVelocity=10, timePerturbationApplication=None):
trajectory_index = 4  # Change this to visualize different trajectories

# Extract probe positions from your data
first_probe_positions = data_sequence['probe_data']['tag_0']['positions']   # HA_TAG positions
second_probe_positions = data_sequence['probe_data']['tag_1']['positions']  # GFP_TAG positions

# Call the function
plot_RibosomeMovement_and_Microscope(
    RibosomePositions=list_ribosome_trajectories[trajectory_index],     # Single trajectory ribosome positions
    IntensityVector=matrix_intensity_first_signal_RT[trajectory_index], # First signal intensity (HA_TAG)
    probePositions=first_probe_positions,                               # HA_TAG probe positions
    SecondIntensityVector=matrix_intensity_second_signal_RT[trajectory_index], # Second signal intensity (GFP_TAG)  
    second_probePositions=second_probe_positions,                       # GFP_TAG probe positions
    fileNameGif='ribosome_movement_traj_0',                            # Output filename
    color='forestgreen',                                                # Color for first signal
    second_color='magenta',                                              # Color for second signal
    FrameVelocity=10,                                                   # Animation speed
    timePerturbationApplication=timePerturbationApplication,                                  # No perturbation
    frame_rate=step_size_in_sec                                        # Frame rate in seconds
)

In [ ]:

# calculating the mean occupancy across all frames
gene_length = data_sequence['gene_length']
ribosomal_density = np.round( (gene_length/global_elongation_rate) *ki , 1)
print(f'Ribosomal density: {ribosomal_density} ribosomes per gene length ({gene_length} codons)')
ribosomal_footprint = 10
ribosomal_density_coverage = np.round( (ribosomal_density * ribosomal_footprint) / gene_length, 2)
percentage_coverage = (ribosomal_density_coverage * 100)  # percentage coverage of the gene length by ribosomes
print(f'Ribosomal density coverage: {np.round(ribosomal_density_coverage,2)} ribosomes per gene length ({gene_length} codons, {percentage_coverage:.2f}% coverage)')
len(list_occupancy_output)
occupancy_array = np.array([np.mean(np.count_nonzero(occupancy, axis=0)) for occupancy in list_occupancy_output])
print(f'Ribosomal density with simulated data: { np.round( np.mean(occupancy_array), 2)} ribosomes per gene length ({gene_length} codons)')


In [ ]:
mean_correlation_ssa_first_signal, std_correlation_ssa_first_signal, lags_ssa, correlations_array, dwell_time = mi.Correlation(primary_data=matrix_intensity_first_signal_RT,   # HA first (reference)
                                                                                                    #secondary_data=matrix_intensity_second_signal_RT, # GFP second (delayed)
                                                                                                    max_lag=None, 
                                                                                                    nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                                    shift_data=True,
                                                                                                    return_full=False,
                                                                                                    time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                                    use_bootstrap=True,
                                                                                                    show_plot=True,
                                                                                                    start_lag=0,
                                                                                                    fit_type='linear',
                                                                                                    de_correlation_threshold=0.1,
                                                                                                    correct_baseline=True,
                                                                                                    use_linear_projection_for_lag_0=False,
                                                                                                    save_plots=False,
                                                                                                    use_global_mean= False,
                                                                                                    remove_outliers = True,
                                                                                                    MAD_THRESHOLD_FACTOR = 6,
                                                                                                    plot_individual_trajectories = False,
                                                                                                    #x_axes_min_max_list_values=[-300,300],
                                                                                                    plot_title=None).run()

In [ ]:
mean_correlation_ssa_second_signal, std_correlation_ssa_second_signal, lags_ssa, correlations_array, dwell_time = mi.Correlation(#primary_data=matrix_intensity_first_signal_RT,   # HA first (reference)
                                                                                                    primary_data=matrix_intensity_second_signal_RT, # GFP second (delayed)
                                                                                                    max_lag=None, 
                                                                                                    nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                                    shift_data=True,
                                                                                                    return_full=False,
                                                                                                    time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                                    use_bootstrap=True,
                                                                                                    show_plot=True,
                                                                                                    start_lag=0,
                                                                                                    fit_type='linear',
                                                                                                    de_correlation_threshold=0.1,
                                                                                                    correct_baseline=True,
                                                                                                    use_linear_projection_for_lag_0=False,
                                                                                                    save_plots=False,
                                                                                                    use_global_mean= False,
                                                                                                    remove_outliers = True,
                                                                                                    MAD_THRESHOLD_FACTOR = 6,
                                                                                                    plot_individual_trajectories = False,
                                                                                                    #x_axes_min_max_list_values=[-300,300],
                                                                                                    plot_title=None).run()

In [ ]:
# plot the acf for both signals using matplotlib
plt.figure(figsize=(8, 5))
plt.plot(lags_ssa, mean_correlation_ssa_first_signal, label='HA_TAG (first signal)', color='forestgreen')
plt.plot(lags_ssa, mean_correlation_ssa_second_signal, label='GFP_TAG (second signal)', color='indigo')
plt.fill_between(lags_ssa, mean_correlation_ssa_first_signal - std_correlation_ssa_first_signal, 
                 mean_correlation_ssa_first_signal + std_correlation_ssa_first_signal, 
                 color='forestgreen', alpha=0.3)
plt.fill_between(lags_ssa, mean_correlation_ssa_second_signal - std_correlation_ssa_second_signal, 
                 mean_correlation_ssa_second_signal + std_correlation_ssa_second_signal, 
                 color='indigo', alpha=0.3)
plt.xlabel('Lag (seconds)')
plt.ylabel('Autocorrelation')
plt.title('Autocorrelation Function (ACF)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Calculate the relationship between decorrelation times and observables
print(f"\n" + "="*60)
print("DECORRELATION TIME → EXPERIMENTAL OBSERVABLES")
print("="*60)

# Effective correlation length in time
ha_correlation_length = 500  # seconds
gfp_correlation_length = 500 / 2.26  # ~221 seconds

print(f"📊 CORRELATION LENGTHS:")
print(f"   • HA_TAG: {ha_correlation_length:.0f}s → events separated by >{ha_correlation_length:.0f}s are independent")
print(f"   • GFP_TAG: {gfp_correlation_length:.0f}s → events separated by >{gfp_correlation_length:.0f}s are independent")

print(f"\n🎯 IMPLICATIONS FOR ANALYSIS:")
print(f"   • Statistical independence: Need >{ha_correlation_length:.0f}s gaps for HA, >{gfp_correlation_length:.0f}s for GFP")
print(f"   • Effective sample size: Total time / decorrelation time")
print(f"   • Noise characterization: Faster decorrelation = more fluctuations")
print(f"   • Model validation: Decorrelation ratio should match position ratio^0.5")

print(f"\n🔬 BIOLOGICAL INSIGHTS:")
print(f"   • Co-translational complexity increases noise accumulation")
print(f"   • Downstream positions are inherently more variable")
print(f"   • Folding delays create additional temporal complexity")
print(f"   • Your model captures realistic molecular dynamics!")

In [ ]:
mean_correlation_ssa, std_correlation_ssa, lags_ssa, correlations_array, dwell_time = mi.Correlation(primary_data=matrix_intensity_first_signal_RT,   # HA first (reference)
                                                                                                    secondary_data=matrix_intensity_second_signal_RT, # GFP second (delayed)
                                                                                                    max_lag=None, 
                                                                                                    nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                                    shift_data=True,
                                                                                                    return_full=True,
                                                                                                    time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                                    use_bootstrap=True,
                                                                                                    show_plot=False,
                                                                                                    start_lag=0,
                                                                                                    fit_type='linear',
                                                                                                    de_correlation_threshold=0.1,
                                                                                                    correct_baseline=True,
                                                                                                    use_linear_projection_for_lag_0=False,
                                                                                                    save_plots=False,
                                                                                                    use_global_mean= False,
                                                                                                    remove_outliers = True,
                                                                                                    MAD_THRESHOLD_FACTOR = 6,
                                                                                                    plot_individual_trajectories = False,
                                                                                                    x_axes_min_max_list_values=[-300,300],
                                                                                                    plot_title=None).run()

In [ ]:
# Expected values based on your parameters
tag_0_positions = data_sequence['probe_data']['tag_0']['positions']  # HA_TAG
tag_1_positions = data_sequence['probe_data']['tag_1']['positions']  # GFP_TAG
print("-" * 40)
print(f"Tag 0 positions (HA_TAG): {tag_0_positions}")
print(f"Tag 1 positions (GFP_TAG): {tag_1_positions}")
print("-" * 40)

avg_distance = np.mean(tag_1_positions) - np.mean(tag_0_positions)
expected_transit_time = avg_distance / global_elongation_rate
expected_lag_frames = expected_transit_time / step_size_in_sec
print("-" * 40)
print(f"Expected distance: {avg_distance:.1f} codons")
print(f"Expected transit time: {expected_transit_time:.1f} seconds") 
print("-" * 40)

# calculate this considering the last position of tag 0 and the first position of tag 1
avg_distance_edge = np.min(tag_1_positions) - np.max(tag_0_positions)
expected_transit_time_edge = avg_distance_edge / global_elongation_rate
expected_lag_frames_edge = expected_transit_time_edge / step_size_in_sec
print("-" * 40)
print(f"Expected distance (edge to edge): {avg_distance_edge:.1f} codons")
print(f"Expected transit time (edge to edge): {expected_transit_time_edge:.1f} seconds")
print("-" * 40)

In [ ]:
def analyze_crosscorr(
    corr,
    dt=5.0,
    lags_s=None,
    zero_index=None,
    window_s=600.0,
    slope_window_pts=3,
    halfmax_fraction=0.5,
    plot=False,
    ax=None,
    plateau_tolerance=0.02,  # New parameter for plateau detection
    sigma_smooth=1,  # Added sigma_smooth parameter
    min_max_normalize=True,  # New parameter for min-max normalization
):
    """
    Analyze a cross-correlation curve to extract delay/shape metrics.
    Now handles flat plateaus by finding the center.
    
    Parameters
    ----------
    corr : (N,) array
        Cross-correlation values vs. lag. Can contain NaNs.
    dt : float, default 5.0
        Sampling interval (seconds per frame).
    lags_s : (N,) array or None
        Lag axis in seconds. If None, built from dt.
    zero_index : int or None
        Index corresponding to lag = 0 s. Defaults to N//2.
    window_s : float, default 600.0
        Time window (±window_s) for area-asymmetry calculation.
    slope_window_pts : int, default 3
        Number of points on each side of 0-lag for local slope fits.
    halfmax_fraction : float, default 0.5
        Fraction of peak for "dominant range" threshold.
    plot : bool, default False
        If True, plots the correlation and annotations.
    ax : matplotlib.axes.Axes or None
        Existing axis to plot into.
    plateau_tolerance : float, default 0.02
        Tolerance for plateau detection (2% below peak).
    sigma_smooth : float, default 1
        Standard deviation for Gaussian smoothing of correlation.
    min_max_normalize : bool, default True
        If True, normalize corr to [0, 1] range based on window_s range.
    """
    corr = np.asarray(corr).astype(float).squeeze()
    if corr.ndim != 1:
        raise ValueError(f"corr must be 1D, got shape {corr.shape}")

    N = len(corr)

    # Build lag axis if not provided
    if lags_s is None:
        if zero_index is None:
            zero_index = N // 2
        lags_frames = np.arange(N) - int(zero_index)
        lags_s = lags_frames * float(dt)
    else:
        lags_s = np.asarray(lags_s).astype(float).squeeze()
        if lags_s.shape != corr.shape:
            raise ValueError("lags_s must have same shape as corr")

    # Interpolate NaNs linearly for stable metrics/plotting
    finite = np.isfinite(corr)
    corr_interp = corr.copy()
    if not np.all(finite):
        good = np.where(finite)[0]
        bad = np.where(~finite)[0]
        if good.size == 0:
            raise ValueError("corr contains no finite values")
        corr_interp[bad] = np.interp(bad, good, corr[good])

    # Apply min-max normalization using ONLY the windowed region
    if min_max_normalize:
        # Define the window mask
        mask = (lags_s >= -window_s) & (lags_s <= window_s)
        
        # Find min/max ONLY within the window
        windowed_data = corr_interp[mask]
        corr_min = np.min(windowed_data)
        corr_max = np.max(windowed_data)
        
        #print(f"Normalization window: ±{window_s} s ({np.sum(mask)} points)")
        print(f"Window range: [{corr_min:.4f}, {corr_max:.4f}]")
        
        if corr_max > corr_min:
            # Apply normalization to entire array using window min/max
            corr_interp = (corr_interp - corr_min) / (corr_max - corr_min)
            print(f"Min-max normalization applied to entire array using window range")
        else:
            # All values in window are the same
            corr_interp = corr_interp - corr_min
            print(f"Warning: Constant correlation values in window, centering at zero")

    # Apply Gaussian smoothing
    corr_smooth = gaussian_filter1d(corr_interp, sigma=sigma_smooth)

    # Peak finding: handle flat plateaus by finding center
    global_max_val = float(np.max(corr_smooth))
    threshold = global_max_val * (1.0 - plateau_tolerance)
    
    # Find all points in plateau
    plateau_mask = corr_smooth >= threshold
    plateau_indices = np.where(plateau_mask)[0]
    
    if len(plateau_indices) > 1:
        # Find center of plateau
        peak_idx = plateau_indices[len(plateau_indices) // 2]
        print(f"Flat plateau detected: {len(plateau_indices)} points, using center at lag {lags_s[peak_idx]:.1f} s")
        print(f"Plateau tolerance: {plateau_tolerance*100:.1f}%, corresponds to {global_max_val*plateau_tolerance:.3f} units below peak value {global_max_val:.3f}")
        print(f"Plateau lag range: {lags_s[plateau_indices[0]]:.1f} s to {lags_s[plateau_indices[-1]]:.1f} s")
    else:
        # Single peak
        peak_idx = plateau_indices[0]

    peak_val = float(corr_smooth[peak_idx])
    peak_lag = float(lags_s[peak_idx])

    # Weighted centroid (shift corr to be nonnegative)
    y_shift = corr_smooth - np.min(corr_smooth)
    denom = y_shift.sum()
    centroid = float(np.sum(lags_s * y_shift) / denom) if denom > 0 else np.nan

    # Half-maximum crossings & FWHM (linear interpolation) - use smoothed data
    def _halfmax_crossings(x, y, frac=0.5):
        p = int(np.argmax(y))
        pk = float(y[p])
        hm = frac * pk
        left_x = np.nan
        for i in range(p, 0, -1):
            y1, y2 = y[i-1], y[i]
            if (y1 - hm) * (y2 - hm) <= 0:
                x1, x2 = x[i-1], x[i]
                left_x = x1 if y2 == y1 else x1 + (hm - y1) * (x2 - x1) / (y2 - y1)
                break
        right_x = np.nan
        for i in range(p, len(y)-1):
            y1, y2 = y[i], y[i+1]
            if (y1 - hm) * (y2 - hm) <= 0:
                x1, x2 = x[i], x[i+1]
                right_x = x2 if y2 == y1 else x1 + (hm - y1) * (x2 - x1) / (y2 - y1)
                break
        width = float(right_x - left_x) if np.isfinite(left_x) and np.isfinite(right_x) else np.nan
        return width, left_x, right_x

    FWHM, left_hm, right_hm = _halfmax_crossings(lags_s, corr_smooth, frac=0.5)

    # Dominant range at specified fraction (contiguous region around peak above threshold)
    def _dominant_range(x, y, frac=0.5):
        p = int(np.argmax(y))
        thresh = frac * float(y[p])
        L = p
        while L-1 >= 0 and y[L-1] >= thresh:
            L -= 1
        R = p
        while R+1 < len(y) and y[R+1] >= thresh:
            R += 1
        return float(x[L]), float(x[R])

    dom_left, dom_right = _dominant_range(lags_s, corr_smooth, frac=halfmax_fraction)

    # Local slopes at 0 (least-squares line fits on small neighborhoods) - use smoothed data
    idx0 = int(np.argmin(np.abs(lags_s - 0.0)))
    L_slice = slice(max(0, idx0 - slope_window_pts), idx0 + 1)
    R_slice = slice(idx0, min(N, idx0 + slope_window_pts + 1))

    def _fit_slope(x, y):
        if len(x) < 2:
            return np.nan
        A = np.vstack([x, np.ones_like(x)]).T
        m, _ = np.linalg.lstsq(A, y, rcond=None)[0]
        return float(m)

    slope_left = _fit_slope(lags_s[L_slice], corr_smooth[L_slice])
    slope_right = _fit_slope(lags_s[R_slice], corr_smooth[R_slice])

    # Area asymmetry in ±window_s (only positive parts) - use smoothed data
    step = float(np.median(np.diff(lags_s)))
    wmask = np.abs(lags_s) <= float(window_s)
    xw = lags_s[wmask]
    yw = corr_smooth[wmask]
    pos_area = float(np.sum(np.clip(yw[xw > 0], 0, None)) * step) if np.any(xw > 0) else 0.0
    neg_area = float(np.sum(np.clip(yw[xw < 0], 0, None)) * step) if np.any(xw < 0) else 0.0
    asym = (pos_area - neg_area) / (pos_area + neg_area + 1e-12)

    metrics = {
        "n_points": N,
        "dt_seconds": float(dt),
        "peak_lag_seconds": peak_lag,
        "peak_value": peak_val,
        "centroid_lag_seconds": centroid,
        "FWHM_seconds": FWHM,
        "FWHM_left_crossing_seconds": left_hm,
        "FWHM_right_crossing_seconds": right_hm,
        "slope_left_at_0_per_s": slope_left,
        "slope_right_at_0_per_s": slope_right,
        "area_asymmetry_pm_window": asym,
        "dominant_left_seconds": dom_left,
        "dominant_right_seconds": dom_right,
    }

    if plot:
        import matplotlib.pyplot as plt
        if ax is None:
            _, ax = plt.subplots(figsize=(8, 4.2))
        
        # Plot both raw and smoothed correlations
        ax.plot(lags_s, corr_interp, lw=1, color="blue", alpha=0.7, label="Raw (normalized)")
        ax.plot(lags_s, corr_smooth, lw=2, color="k", label=f"Smoothed (σ={sigma_smooth})")
        
        ax.axvline(0, ls="--", lw=2, color="gray", alpha=0.5)
        ax.axvline(peak_lag, ls=":", lw=2, color="red", label=f"Peak: {peak_lag:.1f}s")
        
        # Highlight plateau region
        if len(plateau_indices) > 1:
            ax.axvspan(lags_s[plateau_indices[0]], lags_s[plateau_indices[-1]], 
                      alpha=0.2, color="yellow", label=f"Plateau ({len(plateau_indices)} pts)")
        
        # Plot FWHM markers
        if np.isfinite(FWHM):
            ax.axvline(left_hm, ls="-", lw=1, color="red", alpha=0.5)
            ax.axvline(right_hm, ls="-", lw=1, color="red", alpha=0.5)
            ax.hlines(halfmax_fraction * peak_val, left_hm, right_hm, color="red", lw=1, alpha=0.5)
        
        # Set plot limits
        if np.isfinite(window_s):
            ax.set_xlim(-1.1 * window_s, 1.1 * window_s)
        
        if min_max_normalize:
            # For normalized data, show a bit beyond [0,1] for context
            y_min, y_max = ax.get_ylim()
            ax.set_ylim(-0.05, 1.05)
            #ax.set_ylim(min(-0.1, y_min), max(1.1, y_max))
        
        ax.set_title("Cross-correlation (Plateau-aware)")
        ax.set_xlabel("Lag (s)")
        ax.set_ylabel("Cross-correlation (normalized)" if min_max_normalize else "Cross-correlation (a.u.)")
        ax.legend()
        ax.grid(True, alpha=0.2)
        
        if ax is None:
            plt.tight_layout()
            plt.show()

    return metrics, corr_interp, lags_s

# Updated function call with sigma_smooth parameter
max_lag_selected = 700
metrics, corr_interp, lags_s = analyze_crosscorr(
    mean_correlation_ssa, 
    dt=step_size_in_sec, 
    plot=True, 
    window_s=max_lag_selected, 
    slope_window_pts=3, 
    halfmax_fraction=0.5, 
    plateau_tolerance=0.02,
    sigma_smooth=3,  # Add smoothing parameter
    min_max_normalize=True  # Add normalization parameter
)


In [ ]:
raise

In [ ]:
def _find_local_maxima(ref, min_separation_frames=5, max_threshold_fraction=0.80,
                       normalize_for_threshold=True):
    """
    Pure-numpy local-maxima finder with (i) height threshold and (ii) min separation.
    Returns sorted indices (int) of accepted maxima.
    
    Parameters:
    - max_threshold_fraction: minimum relative height (e.g., 0.80 means >= 80% of dynamic range)
    """
    ref = np.asarray(ref, float)
    T = ref.size
    if T < 3:
        return np.array([], dtype=int)

    mid = np.arange(1, T-1)
    # local maxima; allow plateau on right neighbor
    cand = mid[(ref[mid] > ref[mid-1]) & (ref[mid] >= ref[mid+1])]
    if cand.size == 0:
        return cand

    # height threshold (>= fraction of per-trace dynamic range)
    if normalize_for_threshold:
        rmin, rmax = np.nanmin(ref), np.nanmax(ref)
        rng = (rmax - rmin) if np.isfinite(rmax - rmin) and (rmax > rmin) else 1.0
        ref_norm = (ref - rmin) / rng
        keep = ref_norm[cand] >= float(max_threshold_fraction)
    else:
        keep = ref[cand] >= float(max_threshold_fraction)
    cand = cand[keep]
    if cand.size <= 1 or min_separation_frames <= 1:
        return np.sort(cand)

    # enforce minimum separation (keep higher maxima first)
    order = np.argsort(-ref[cand])  # descending (higher first)
    kept = []
    banned = np.zeros(T, dtype=bool)
    for idx in cand[order]:
        if not banned[idx]:
            kept.append(idx)
            lo = max(0, idx - min_separation_frames)
            hi = min(T, idx + min_separation_frames + 1)
            banned[lo:hi] = True
    return np.array(sorted(kept), dtype=int)

def align_by_maxima_two_channel(
    first_matrix,                  # (N_traces, T)
    second_matrix,                 # (N_traces, T)
    *,
    reference="first",             # "first" or "second"
    dt_seconds=5.0,
    window_frames=10,              # half-window size -> total length = 2*window_frames+1
    min_separation_frames=6,
    max_threshold_fraction=0.80,   # minimum height threshold (e.g., 0.80 = top 20% of signal range)
    normalize_for_threshold=True,
    max_events_per_trace=None,     # e.g., 3; None = no cap
    skip_nan_windows=True,
    plot=True,
    plot_individual_trajectories=False, # NEW: Plot individual trajectories alongside averages
    title_prefix="Max-sync (2-ch)",
    normalize_each_trajectory=True, # NEW: Min-max normalize each individual trajectory/window
    normalize_averaged_signals=False, # Normalize the final averaged signals
    detect_maxima_in_windows=True  # Detect maxima (True) or minima (False) in extracted windows
):
    """
    Align two-channel signals by detecting maxima in reference channel.
    Each individual trajectory can be min-max normalized before averaging.
    
    Parameters:
    -----------
    plot_individual_trajectories : bool (default False)
        Plot all individual extracted trajectories with transparency alongside the averaged signals
    normalize_each_trajectory : bool (default True)
        Min-max normalize each individual extracted window/trajectory to [0,1] before averaging
    normalize_averaged_signals : bool (default False)  
        Additionally normalize the final averaged signals to [0,1]
    
    Returns:
    --------
      matrix_intensity_first_signal_RT  : (N_events, 2*window_frames+1)
      matrix_intensity_second_signal_RT : (N_events, 2*window_frames+1)
      time_axis_seconds                 : (2*window_frames+1,)
      extras (dict) with counts and delay information
    """
    first_matrix  = np.asarray(first_matrix, float)
    second_matrix = np.asarray(second_matrix, float)
    assert first_matrix.shape == second_matrix.shape, "Both channels must have shape (N_traces, T)"
    N, T = first_matrix.shape

    # Always use first channel as reference for maxima detection (as per user requirement)
    ref_mat = first_matrix
    print(f"Using first channel as reference for maxima detection")
    print(f"Normalization: each_trajectory={normalize_each_trajectory}, averaged_signals={normalize_averaged_signals}")
    print(f"Individual trajectories: plot={plot_individual_trajectories}")

    win = int(window_frames)
    L = 2 * win + 1
    tsec = np.arange(-win, win + 1, dtype=float) * float(dt_seconds)

    windows_first, windows_second, ref_windows = [], [], []
    event_offsets_frames = []  # max position within window relative to center

    for i in range(N):
        ref = ref_mat[i, :]

        # Find maxima in the reference (first) channel
        maxs = _find_local_maxima(
            ref,
            min_separation_frames=min_separation_frames,
            max_threshold_fraction=max_threshold_fraction,
            normalize_for_threshold=normalize_for_threshold
        )
        if maxs.size == 0:
            continue

        if max_events_per_trace is not None and maxs.size > max_events_per_trace:
            # choose highest maxima
            order = np.argsort(-ref[maxs])  # highest first
            maxs = np.sort(maxs[order[:max_events_per_trace]])

        for m in maxs:
            lo = m - win
            hi = m + win + 1
            if lo < 0 or hi > T:
                continue  # skip events too close to edges

            w_ref   = ref_mat[i, lo:hi]
            w_first = first_matrix[i, lo:hi]
            w_second= second_matrix[i, lo:hi]

            if skip_nan_windows and (np.any(~np.isfinite(w_first)) or np.any(~np.isfinite(w_second))):
                continue

            windows_first.append(w_first)
            windows_second.append(w_second)
            ref_windows.append(w_ref)

            # compute max offset in frames within the window (robust to plateaus)
            # pick center index of plateau near the global max in the window
            w = w_ref
            vmax = np.nanmax(w)
            plateau = np.where(np.isclose(w, vmax, rtol=0, atol=1e-12))[0]
            if plateau.size > 0:
                max_idx = int(plateau[plateau.size // 2])
            else:
                max_idx = int(np.nanargmax(w))
            event_offsets_frames.append(max_idx - win)

    if len(windows_first) == 0:
        # empty result with correct shapes
        return (np.empty((0, L), float),
                np.empty((0, L), float),
                tsec,
                {"n_traces": N, "n_events": 0, "avg_max_offset_seconds": np.nan})

    matrix_intensity_first_signal_RT  = np.vstack(windows_first)
    matrix_intensity_second_signal_RT = np.vstack(windows_second)
    ref_windows = np.vstack(ref_windows)

    avg_max_offset_frames = float(np.mean(event_offsets_frames)) if len(event_offsets_frames) else 0.0
    avg_max_offset_seconds = avg_max_offset_frames * float(dt_seconds)

    # Calculate statistics for both channels
    m1 = np.nanmean(matrix_intensity_first_signal_RT, axis=0)
    s1 = np.nanstd(matrix_intensity_first_signal_RT, axis=0)
    sem1 = s1 / np.sqrt(matrix_intensity_first_signal_RT.shape[0]) if matrix_intensity_first_signal_RT.shape[0] > 1 else s1

    m2 = np.nanmean(matrix_intensity_second_signal_RT, axis=0)
    s2 = np.nanstd(matrix_intensity_second_signal_RT, axis=0)
    sem2 = s2 / np.sqrt(matrix_intensity_second_signal_RT.shape[0]) if matrix_intensity_second_signal_RT.shape[0] > 1 else s2

    # Optional signal normalization of averaged signals
    if normalize_averaged_signals:
        # Normalize each signal to [0,1] range
        m1_norm = (m1 - np.nanmin(m1)) / (np.nanmax(m1) - np.nanmin(m1)) if np.nanmax(m1) > np.nanmin(m1) else m1
        m2_norm = (m2 - np.nanmin(m2)) / (np.nanmax(m2) - np.nanmin(m2)) if np.nanmax(m2) > np.nanmin(m2) else m2
        sem1_norm = sem1 / (np.nanmax(m1) - np.nanmin(m1)) if np.nanmax(m1) > np.nanmin(m1) else sem1
        sem2_norm = sem2 / (np.nanmax(m2) - np.nanmin(m2)) if np.nanmax(m2) > np.nanmin(m2) else sem2
        m1, m2, sem1, sem2 = m1_norm, m2_norm, sem1_norm, sem2_norm

    # Find features in averaged signals for delay calculation
    if detect_maxima_in_windows:
        # Look for maxima in both averaged signals
        first_feature_idx = np.nanargmax(m1)
        second_feature_idx = np.nanargmax(m2)
        feature_type = "maxima"
        print(f"Detecting MAXIMA in averaged windows")
    else:
        # Look for minima in both averaged signals
        first_feature_idx = np.nanargmin(m1)
        second_feature_idx = np.nanargmin(m2)
        feature_type = "minima"
        print(f"Detecting MINIMA in averaged windows")
    
    first_feature_time = tsec[first_feature_idx]
    second_feature_time = tsec[second_feature_idx]
    feature_delay = second_feature_time - first_feature_time

    print(f"First channel {feature_type[:-1]} value: {m1[first_feature_idx]:.3f} at time {first_feature_time:.1f}s")
    print(f"Second channel {feature_type[:-1]} value: {m2[second_feature_idx]:.3f} at time {second_feature_time:.1f}s")
    print(f"Delay between {feature_type}: {feature_delay:.1f}s")
    
    if plot and plot_individual_trajectories:
        traj_type = "normalized" if normalize_each_trajectory else "raw"
        print(f"Individual trajectories will be plotted using {traj_type} data")

    # ---------- plotting ----------
    if plot:
        # Create combined plot
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))

        # Plot individual trajectories if requested
        if plot_individual_trajectories:
            n_events = matrix_intensity_first_signal_RT.shape[0]
            # Limit number of individual trajectories shown to avoid clutter
            max_individual_plots = min(50, n_events)  # Show max 50 trajectories
            indices = np.linspace(0, n_events-1, max_individual_plots, dtype=int) if n_events > max_individual_plots else range(n_events)
            
            # Note: These trajectories are already normalized if normalize_each_trajectory=True
            # because matrix_intensity_*_signal_RT contains the processed (normalized) windows
            for i in indices:
                ax.plot(tsec, matrix_intensity_first_signal_RT[i], 
                       color='forestgreen', alpha=0.1, lw=0.5)
                ax.plot(tsec, matrix_intensity_second_signal_RT[i], 
                       color='purple', alpha=0.1, lw=0.5)
            
            # Add legend entries for individual trajectories
            traj_status = " (normalized)" if normalize_each_trajectory else " (raw)"
            ax.plot([], [], color='forestgreen', alpha=0.3, lw=1, 
                   label=f"First individual{traj_status} (n={n_events})")
            ax.plot([], [], color='purple', alpha=0.3, lw=1, 
                   label=f"Second individual{traj_status} (n={n_events})")

        # Plot both channels on same axis (averages)
        line1 = ax.plot(tsec, m1, lw=2, color='forestgreen', label=f"First channel (mean)", alpha=0.9)
        ax.fill_between(tsec, m1 - sem1, m1 + sem1, color='forestgreen', alpha=0.2)
        
        line2 = ax.plot(tsec, m2, lw=2, color='purple', label=f"Second channel (mean)", alpha=0.9)
        ax.fill_between(tsec, m2 - sem2, m2 + sem2, color='purple', alpha=0.2)

        # Mark reference point (average maximum offset from reference channel)
        ax.axvline(avg_max_offset_seconds, ls="--", lw=2, color="black", alpha=0.7, 
                  label=f"Ref max offset: {avg_max_offset_seconds:.1f}s")
        
        # Mark zero lag
        ax.axvline(0, ls=":", lw=1, color="gray", alpha=0.5, label="Zero lag")
        
        # Mark detected features and delay
        ax.axvline(first_feature_time, ls="-", lw=1, color="forestgreen", alpha=0.7)
        ax.axvline(second_feature_time, ls="-", lw=1, color="purple", alpha=0.7)
        
        # Mark the actual feature points
        ax.plot(first_feature_time, m1[first_feature_idx], 'o', color='forestgreen', 
                markersize=8, markeredgecolor='white', markeredgewidth=2, 
                label=f"First {feature_type[:-1]}")
        ax.plot(second_feature_time, m2[second_feature_idx], 'o', color='purple', 
                markersize=8, markeredgecolor='white', markeredgewidth=2,
                label=f"Second {feature_type[:-1]}")
        
        # Add delay annotation
        if abs(feature_delay) > dt_seconds/2:  # Only show if delay is significant
            y_level = max(np.nanmax(m1), np.nanmax(m2)) * 0.9
            ax.annotate('', xy=(second_feature_time, y_level), 
                       xytext=(first_feature_time, y_level),
                       arrowprops=dict(arrowstyle='<->', color='red', lw=2))
            ax.text((first_feature_time + second_feature_time)/2, 
                   y_level + max(np.nanmax(m1), np.nanmax(m2)) * 0.05,
                   f'Delay: {feature_delay:.1f}s', 
                   ha='center', va='bottom', color='red', fontweight='bold')

        title_suffix = " (with individual trajectories)" if plot_individual_trajectories else ""
        ax.set_title(f"{title_prefix}: Event-triggered average ({feature_type} detection){title_suffix}\n"
                    f"n_events={matrix_intensity_first_signal_RT.shape[0]}, "
                    f"Feature delay: {feature_delay:.1f}s")
        ax.set_xlabel("Time relative to reference maximum (s)")
        
        # Update y-axis label based on normalization settings
        if normalize_each_trajectory and normalize_averaged_signals:
            ylabel = "Intensity (trajectory + average normalized)"
        elif normalize_each_trajectory:
            ylabel = "Intensity (trajectory normalized)"
        elif normalize_averaged_signals:
            ylabel = "Intensity (average normalized)"
        else:
            ylabel = "Intensity (a.u.)"
        ax.set_ylabel(ylabel)
        
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

        # Also create separate subplot version for detailed view
        fig2, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True, sharey=True)
        
        # Plot individual trajectories in subplots if requested
        if plot_individual_trajectories:
            n_events = matrix_intensity_first_signal_RT.shape[0]
            max_individual_plots = min(30, n_events)  # Show fewer in subplots
            indices = np.linspace(0, n_events-1, max_individual_plots, dtype=int) if n_events > max_individual_plots else range(n_events)
            
            # First channel subplot - plot normalized trajectories if normalize_each_trajectory=True
            for i in indices:
                axes[0].plot(tsec, matrix_intensity_first_signal_RT[i], 
                           color='forestgreen', alpha=0.15, lw=0.5)
            
            # Second channel subplot - plot normalized trajectories if normalize_each_trajectory=True
            for i in indices:
                axes[1].plot(tsec, matrix_intensity_second_signal_RT[i], 
                           color='purple', alpha=0.15, lw=0.5)
        
        # First channel
        axes[0].plot(tsec, m1, lw=2, color='forestgreen', label="First (mean)")
        axes[0].fill_between(tsec, m1 - sem1, m1 + sem1, color='forestgreen', alpha=0.2)
        axes[0].axvline(avg_max_offset_seconds, ls="--", lw=2, color="black", alpha=0.7)
        axes[0].axvline(first_feature_time, ls="-", lw=1, color="forestgreen", alpha=0.7)
        axes[0].plot(first_feature_time, m1[first_feature_idx], 'o', color='forestgreen', markersize=8, markeredgecolor='white', markeredgewidth=2)
        
        title_suffix = f" + {n_events} individual" if plot_individual_trajectories else ""
        traj_norm_suffix = " (norm)" if plot_individual_trajectories and normalize_each_trajectory else ""
        axes[0].set_title(f"First channel (Reference){title_suffix}{traj_norm_suffix}")
        axes[0].set_xlabel("Time (s)")
        
        # Consistent y-axis labeling
        if normalize_each_trajectory and normalize_averaged_signals:
            ylabel = "Intensity (traj + avg norm)"
        elif normalize_each_trajectory:
            ylabel = "Intensity (traj norm)"
        elif normalize_averaged_signals:
            ylabel = "Intensity (avg norm)"
        else:
            ylabel = "Intensity"
        axes[0].set_ylabel(ylabel)
        
        axes[0].grid(True, alpha=0.2)
        axes[0].legend()

        # Second channel
        axes[1].plot(tsec, m2, lw=2, color='purple', label="Second (mean)")
        axes[1].fill_between(tsec, m2 - sem2, m2 + sem2, color='purple', alpha=0.2)
        axes[1].axvline(avg_max_offset_seconds, ls="--", lw=2, color="black", alpha=0.7)
        axes[1].axvline(second_feature_time, ls="-", lw=1, color="purple", alpha=0.7)
        axes[1].plot(second_feature_time, m2[second_feature_idx], 'o', color='purple', markersize=8, markeredgecolor='white', markeredgewidth=2)
        axes[1].set_title(f"Second channel (Expected delay){title_suffix}{traj_norm_suffix}")
        axes[1].set_xlabel("Time (s)")
        axes[1].grid(True, alpha=0.2)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

    extras = {
        "n_traces": N,
        "n_events": int(matrix_intensity_first_signal_RT.shape[0]),
        "avg_max_offset_seconds": avg_max_offset_seconds,
        "reference_channel": "first",  # Always first as per requirements
        "feature_delay_seconds": feature_delay,
        "first_feature_time": first_feature_time,
        "second_feature_time": second_feature_time,
        "first_channel_stats": {"mean": m1, "sem": sem1},
        "second_channel_stats": {"mean": m2, "sem": sem2},
        "alignment_method": "maxima_based",
        "feature_detection_method": feature_type,
        "feature_detection_values": {
            "first_channel": m1[first_feature_idx],
            "second_channel": m2[second_feature_idx]
        },
        "max_threshold_fraction": max_threshold_fraction,
        "normalization_settings": {
            "normalize_each_trajectory": normalize_each_trajectory,
            "normalize_averaged_signals": normalize_averaged_signals
        },
        "plot_settings": {
            "plot_individual_trajectories": plot_individual_trajectories,
            "plot_enabled": plot
        }
    }
    
    return (matrix_intensity_first_signal_RT,
            matrix_intensity_second_signal_RT,
            tsec,
            extras)

In [ ]:
# For maxima detection and alignment
result = align_by_maxima_two_channel(
    matrix_intensity_first_signal_RT,
    matrix_intensity_second_signal_RT, 
    max_threshold_fraction=0.8,      # Top 20% of signal range
    detect_maxima_in_windows=True,    # Find maxima in averaged windows
    reference="first",                # Use first channel for maxima detection
    window_frames=20,                 # Window size
    min_separation_frames=6,          # Minimum separation between events
    plot=True,
    plot_individual_trajectories=True,
    #normalize_each_trajectory=True, 
    #normalize_averaged_signals=True 
)

In [ ]:
def _running_median(x, w):
    if w <= 1 or w > len(x): return x.copy()
    y = x.copy(); h = w//2
    for i in range(len(x)):
        y[i] = np.nanmedian(x[max(0, i-h):min(len(x), i+h+1)])
    return y

def _find_rising_edges(y, smooth_win=7, amp_q=0.7, min_sep=1):
    """Return rising-edge indices in y using a quantile threshold on the positive derivative."""
    ys = _running_median(np.asarray(y, float), int(smooth_win))
    dy = np.diff(ys, prepend=ys[0])
    pos = dy[dy > 0]
    if pos.size == 0:
        return np.array([], dtype=int), ys, dy, np.inf
    thr = float(np.quantile(pos, amp_q))
    idx = np.where(dy >= thr)[0]
    # enforce minimum separation
    keep = []
    last = -min_sep
    for k in idx:
        if k - last >= min_sep:
            keep.append(k); last = k
    return np.array(keep, int), ys, dy, thr

def _event_windows(x, centers, halfwin):
    """Stack windows of x centered at 'centers' (indices), skipping edges."""
    L = 2*halfwin + 1
    out = []
    for c in centers:
        lo, hi = c - halfwin, c + halfwin + 1
        if lo >= 0 and hi <= len(x):
            w = x[lo:hi]
            if np.all(np.isfinite(w)):
                out.append(w)
    return np.vstack(out) if out else np.empty((0, L), float)

def plot_delay_dashboard(
    first_matrix, second_matrix, dt,
    reference="first", smooth_win=7,
    min_separation_s=60.0, amp_q=0.7,
    search_window_s=240.0, window_s=180.0,
    example_trace_index=None
):
    """
    first_matrix, second_matrix : arrays (n_traces, T)
    dt : seconds per frame
    reference : 'first' or 'second' (which channel to detect edges on)
    smooth_win : running-median window (odd int)
    min_separation_s : min gap between accepted ref edges (seconds)
    amp_q : edge threshold = amp_q-quantile of positive dI/dt
    search_window_s : lookahead to find other-channel edge after a ref edge
    window_s : half-window for edge-aligned averaging (seconds)
    """
    A = np.asarray(first_matrix, float)
    B = np.asarray(second_matrix, float)
    assert A.shape == B.shape, "Both channels must have shape (n_traces, T)"
    n, T = A.shape
    use_A = (reference.lower() == "first")
    REF, OTH = (A, B) if use_A else (B, A)

    halfwin = int(round(window_s / float(dt)))
    min_sep = max(1, int(round(min_separation_s / float(dt))))
    look    = max(1, int(round(search_window_s   / float(dt))))
    t_edge  = np.arange(-halfwin, halfwin+1) * float(dt)

    delays = []
    ref_edge_indices = []
    oth_edge_indices = []
    trace_ids = []

    # detect edges and delays per trace
    for i in range(n):
        idx_ref, ref_smooth, _, _ = _find_rising_edges(
            REF[i], smooth_win=smooth_win, amp_q=amp_q, min_sep=min_sep
        )
        if idx_ref.size == 0:
            continue
        idx_oth, _, _, _ = _find_rising_edges(
            OTH[i], smooth_win=smooth_win, amp_q=amp_q, min_sep=1
        )
        if idx_oth.size == 0:
            continue

        for k in idx_ref:
            # first other edge after ref edge within lookahead window
            cand = idx_oth[(idx_oth > k) & (idx_oth <= k + look)]
            if cand.size:
                j = int(cand[0])
                delays.append((j - k) * float(dt))
                ref_edge_indices.append(k)
                oth_edge_indices.append(j)
                trace_ids.append(i)

    delays = np.array(delays, float)
    if delays.size == 0:
        print("No delays detected (check thresholds or increase search_window_s).")
        return {"delays": delays, "median_delay_s": np.nan, "n_events": 0}

    median_delay = float(np.nanmedian(delays))

    # build edge-aligned windows for both channels
    W_first = []
    W_second = []
    for i, k in zip(trace_ids, ref_edge_indices):
        W_first.append(_event_windows(A[i], [k], halfwin))
        W_second.append(_event_windows(B[i], [k], halfwin))
    W_first  = np.vstack(W_first)  if W_first  else np.empty((0, 2*halfwin+1), float)
    W_second = np.vstack(W_second) if W_second else np.empty((0, 2*halfwin+1), float)

    # choose an example event to show (first by default)
    ex = 0 if example_trace_index is None else int(example_trace_index)
    if 0 <= ex < len(trace_ids):
        ex_tid = trace_ids[ex]
        ex_k   = ref_edge_indices[ex]
        ex_j   = oth_edge_indices[ex]
    else:
        ex_tid, ex_k, ex_j = trace_ids[0], ref_edge_indices[0], oth_edge_indices[0]

    # ---------- plotting ----------
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    # (1) Example trace with detected edges and delay arrow
    t = np.arange(T) * float(dt)
    axes[0].plot(t, A[ex_tid], lw=1.6, label="First")
    axes[0].plot(t, B[ex_tid], lw=1.6, label="Second")
    axes[0].axvline(ex_k*dt, ls="--")
    axes[0].axvline(ex_j*dt, ls="--")
    axes[0].annotate("", xy=(ex_j*dt, np.nanmax([A[ex_tid], B[ex_tid]])*0.9),
                     xytext=(ex_k*dt, np.nanmax([A[ex_tid], B[ex_tid]])*0.9),
                     arrowprops=dict(arrowstyle="<->"))
    axes[0].set_title(f"Example trace (delay ≈ {(ex_j-ex_k)*dt:.1f}s)")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Intensity (a.u.)")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    # (2) Edge-aligned means (vertical line at median delay)
    m1 = np.nanmean(W_first, axis=0)  if W_first.size  else np.zeros_like(t_edge)
    s1 = np.nanstd(W_first,  axis=0)  if W_first.size  else np.zeros_like(t_edge)
    m2 = np.nanmean(W_second, axis=0) if W_second.size else np.zeros_like(t_edge)
    s2 = np.nanstd(W_second,  axis=0) if W_second.size else np.zeros_like(t_edge)

    axes[1].plot(t_edge, m1, lw=2, label="First (ref edge @ 0)")
    axes[1].fill_between(t_edge, m1 - s1, m1 + s1, alpha=0.2, linewidth=0)
    axes[1].plot(t_edge, m2, lw=2, label="Second")
    axes[1].fill_between(t_edge, m2 - s2, m2 + s2, alpha=0.2, linewidth=0)
    axes[1].axvline(0.0, ls="--")
    axes[1].axvline(median_delay, ls=":")
    axes[1].set_title(f"Edge-aligned averages (median delay ≈ {median_delay:.1f}s)")
    axes[1].set_xlabel("Time from ref edge (s)")
    axes[1].grid(alpha=0.2)
    axes[1].legend()

    # (3) Delay distribution
    axes[2].hist(delays, bins=max(8, int(np.sqrt(delays.size))))
    axes[2].axvline(median_delay, ls="--")
    axes[2].set_title("Per-event delay distribution")
    axes[2].set_xlabel("Delay (s)")
    axes[2].set_ylabel("Count")
    axes[2].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

    return {
        "delays": delays,
        "median_delay_s": median_delay,
        "n_events": delays.size,
        "edge_aligned_first": W_first,
        "edge_aligned_second": W_second,
        "edge_time_axis_s": t_edge,
    }

In [ ]:
# first_matrix, second_matrix : shape (n_traces, T)
out = plot_delay_dashboard(
    matrix_intensity_first_signal_RT,
    matrix_intensity_second_signal_RT, 
    dt=5.0,
    reference="first",        # or "second"
    smooth_win=7,             # adjust if noisy
    min_separation_s=60.0,    # avoid counting closely spaced steps twice
    amp_q=0.7,                # 70th pct of positive dI/dt as edge threshold
    search_window_s=240.0,    # must exceed expected lag
    window_s=180.0            # half-window for edge-aligned averaging
)

print(f"Events: {out['n_events']}, median delay: {out['median_delay_s']:.1f}s")

In [ ]:
def _running_median(x, w):
    x = np.asarray(x, float)
    if w <= 1 or w > x.size: 
        return x.copy()
    y = x.copy(); h = w//2
    for i in range(x.size):
        y[i] = np.nanmedian(x[max(0, i-h):min(x.size, i+h+1)])
    return y

def _find_rising_edges(y, smooth_win=7, amp_q=0.7, min_sep=1):
    """
    Rising-edge detector: median-smooth y, threshold on positive derivative via a quantile.
    Returns: indices (int), smoothed_signal, derivative, threshold_used
    """
    ys = _running_median(np.asarray(y, float), int(smooth_win))
    dy = np.diff(ys, prepend=ys[0])
    pos = dy[dy > 0]
    if pos.size == 0:
        return np.array([], dtype=int), ys, dy, np.inf
    thr = float(np.quantile(pos, amp_q))
    idx = np.where(dy >= thr)[0]
    # enforce minimum separation
    keep, last = [], -min_sep
    for k in idx:
        if k - last >= min_sep:
            keep.append(k); last = k
    return np.array(keep, int), ys, dy, thr

def _event_windows(x, centers, halfwin):
    """
    Stack windows of x centered at 'centers' (indices), skipping edges/NaNs.
    """
    x = np.asarray(x, float)
    L = 2*halfwin + 1
    out = []
    for c in centers:
        lo, hi = c - halfwin, c + halfwin + 1
        if lo >= 0 and hi <= x.size:
            w = x[lo:hi]
            if np.all(np.isfinite(w)):
                out.append(w)
    return np.vstack(out) if out else np.empty((0, L), float)

def plot_delay_dashboard(
    first_matrix, second_matrix, dt,
    reference="first", smooth_win=7,
    min_separation_s=60.0, amp_q=0.7,
    search_window_s=240.0, window_s=180.0,
    example_trace_index=None
):
    """
    Make a 3-panel visualization and return delays and edge-aligned windows.
    """
    A = np.asarray(first_matrix, float)
    B = np.asarray(second_matrix, float)
    assert A.shape == B.shape, "Both channels must have shape (n_traces, T)"
    n, T = A.shape
    use_A = (reference.lower() == "first")
    REF, OTH = (A, B) if use_A else (B, A)

    halfwin = int(round(window_s / float(dt)))
    min_sep = max(1, int(round(min_separation_s / float(dt))))
    look    = max(1, int(round(search_window_s   / float(dt))))
    t_edge  = np.arange(-halfwin, halfwin+1) * float(dt)

    delays = []
    ref_edge_indices = []
    oth_edge_indices = []
    trace_ids = []

    # detect edges and per-event delays
    for i in range(n):
        idx_ref, _, _, _ = _find_rising_edges(REF[i], smooth_win=smooth_win, amp_q=amp_q, min_sep=min_sep)
        if idx_ref.size == 0:
            continue
        idx_oth, _, _, _ = _find_rising_edges(OTH[i], smooth_win=smooth_win, amp_q=amp_q, min_sep=1)
        if idx_oth.size == 0:
            continue

        for k in idx_ref:
            cand = idx_oth[(idx_oth > k) & (idx_oth <= k + look)]
            if cand.size:
                j = int(cand[0])
                delays.append((j - k) * float(dt))
                ref_edge_indices.append(k)
                oth_edge_indices.append(j)
                trace_ids.append(i)

    delays = np.array(delays, float)
    median_delay = float(np.nanmedian(delays)) if delays.size else np.nan

    # build edge-aligned windows
    W_first = []
    W_second = []
    for i, k in zip(trace_ids, ref_edge_indices):
        W_first.append(_event_windows(A[i], [k], halfwin))
        W_second.append(_event_windows(B[i], [k], halfwin))
    W_first  = np.vstack(W_first)  if W_first  else np.empty((0, 2*halfwin+1), float)
    W_second = np.vstack(W_second) if W_second else np.empty((0, 2*halfwin+1), float)

    # pick an example event
    ex = 0 if (example_trace_index is None or example_trace_index >= len(trace_ids)) else int(example_trace_index)
    if len(trace_ids):
        ex_tid, ex_k, ex_j = trace_ids[ex], ref_edge_indices[ex], oth_edge_indices[ex]
    else:
        ex_tid, ex_k, ex_j = 0, 0, 0

    # ---- plotting ----
    fig = plt.figure(figsize=(15, 4.2))

    # 1) Example trace (own axes)
    ax1 = fig.add_axes([0.06, 0.15, 0.28, 0.75])
    t = np.arange(T) * float(dt)
    if len(trace_ids):
        ax1.plot(t, A[ex_tid], lw=1.6, label="First")
        ax1.plot(t, B[ex_tid], lw=1.6, label="Second")
        ax1.axvline(ex_k*dt, ls="--")
        ax1.axvline(ex_j*dt, ls="--")
        ymax = max(np.nanmax(A[ex_tid]), np.nanmax(B[ex_tid]))
        ax1.annotate("", xy=(ex_j*dt, 0.9*ymax), xytext=(ex_k*dt, 0.9*ymax),
                     arrowprops=dict(arrowstyle="<->"))
        ax1.set_title(f"Example trace (delay ≈ {(ex_j-ex_k)*dt:.1f}s)")
    else:
        ax1.set_title("Example trace (no edges found)")
    ax1.set_xlabel("Time (s)"); ax1.set_ylabel("Intensity (a.u.)")
    ax1.grid(alpha=0.2); ax1.legend(loc="upper left")

    # 2) Edge-aligned means
    ax2 = fig.add_axes([0.38, 0.15, 0.28, 0.75])
    m1 = np.nanmean(W_first, axis=0)  if W_first.size  else np.zeros_like(t_edge)
    s1 = np.nanstd(W_first,  axis=0)  if W_first.size  else np.zeros_like(t_edge)
    m2 = np.nanmean(W_second, axis=0) if W_second.size else np.zeros_like(t_edge)
    s2 = np.nanstd(W_second,  axis=0) if W_second.size else np.zeros_like(t_edge)
    ax2.plot(t_edge, m1, lw=2, label="First (ref edge @ 0)")
    ax2.fill_between(t_edge, m1 - s1, m1 + s1, alpha=0.2, linewidth=0)
    ax2.plot(t_edge, m2, lw=2, label="Second")
    ax2.fill_between(t_edge, m2 - s2, m2 + s2, alpha=0.2, linewidth=0)
    ax2.axvline(0.0, ls="--")
    if np.isfinite(median_delay):
        ax2.axvline(median_delay, ls=":")
        ax2.set_title(f"Edge-aligned averages (median delay ≈ {median_delay:.1f}s)")
    else:
        ax2.set_title("Edge-aligned averages (no delays)")
    ax2.set_xlabel("Time from ref edge (s)")
    ax2.grid(alpha=0.2); ax2.legend()

    # 3) Delay histogram
    ax3 = fig.add_axes([0.70, 0.15, 0.28, 0.75])
    if delays.size:
        bins = max(8, int(np.sqrt(delays.size)))
        ax3.hist(delays, bins=bins)
        ax3.axvline(median_delay, ls="--")
        ax3.set_title("Per-event delay distribution")
    else:
        ax3.set_title("Per-event delay distribution (none)")
    ax3.set_xlabel("Delay (s)"); ax3.set_ylabel("Count")
    ax3.grid(alpha=0.2)

    plt.show()

    return {
        "delays": delays,
        "median_delay_s": median_delay,
        "n_events": delays.size,
        "edge_aligned_first": W_first,
        "edge_aligned_second": W_second,
        "edge_time_axis_s": t_edge,
    }


rng = np.random.default_rng(7)

def synth_dataset_fixed_delay(n_traces=20, T=1200, dt=5.0,
                              mean_step_every_s=90.0,
                              fixed_delay_s=50.0,
                              step_height=0.08, noise_sd=0.02,
                              bleach_tau_s=np.inf):
    """
    Piecewise-constant stairs with a fixed delay from channel1 -> channel2.
    """
    A = np.zeros((n_traces, T), float)
    B = np.zeros((n_traces, T), float)
    lam = 1.0 / max(1e-6, mean_step_every_s)
    for i in range(n_traces):
        t = 0.0
        a = np.zeros(T, float); b = np.zeros(T, float)
        while True:
            dt_event = rng.exponential(1/lam)  # seconds to next event
            t += dt_event
            k = int(round(t / dt))
            if k >= T: break
            a[k:] += step_height
            # delayed event in B
            kd = int(round((t + fixed_delay_s) / dt))
            if kd < T:
                b[kd:] += step_height
        # optional exponential bleaching (same on both)
        if np.isfinite(bleach_tau_s):
            decay = np.exp(-(np.arange(T)*dt) / bleach_tau_s)
            a *= decay; b *= decay
        # noise
        a += rng.normal(0, noise_sd, size=T)
        b += rng.normal(0, noise_sd, size=T)
        # normalize to [0,1]
        for arr, out in [(a, A[i]), (b, B[i])]:
            lo, hi = np.nanmin(arr), np.nanmax(arr)
            out[:] = (arr - lo) / (hi - lo + 1e-12)
    return A, B

def synth_dataset_variable_delay(n_traces=20, T=1200, dt=5.0,
                                 mean_step_every_s=90.0,
                                 delay_mean_s=50.0, delay_sd_s=25.0,
                                 step_height=0.08, noise_sd=0.03,
                                 missing_prob=0.15,
                                 common_drift_amp=0.15, drift_period_s=1200.0,
                                 bleach_tau_s=1800.0):
    """
    Messy, TASEP-like variability: per-event random delay, shared drift, dropouts, noise.
    """
    A = np.zeros((n_traces, T), float)
    B = np.zeros((n_traces, T), float)
    lam = 1.0 / max(1e-6, mean_step_every_s)
    tt = np.arange(T) * dt
    drift = common_drift_amp * (0.5 + 0.5*np.sin(2*np.pi*tt / drift_period_s))  # shared slow drift
    for i in range(n_traces):
        t = 0.0
        a = np.zeros(T, float); b = np.zeros(T, float)
        while True:
            dt_event = rng.exponential(1/lam)
            t += dt_event
            k = int(round(t / dt))
            if k >= T: break
            a[k:] += step_height
            # variable delay for B
            d = max(0.0, rng.normal(delay_mean_s, delay_sd_s))
            kd = int(round((t + d) / dt))
            if kd < T and rng.random() > missing_prob:
                b[kd:] += step_height
        # add drift, bleaching, and noise
        a = a * (np.exp(-tt / bleach_tau_s)) + drift
        b = b * (np.exp(-tt / bleach_tau_s)) + drift
        a += rng.normal(0, noise_sd, size=T)
        b += rng.normal(0, noise_sd, size=T)
        # normalize to [0,1]
        for arr, out in [(a, A[i]), (b, B[i])]:
            lo, hi = np.nanmin(arr), np.nanmax(arr)
            out[:] = (arr - lo) / (hi - lo + 1e-12)
    return A, B


if __name__ == "__main__":
    dt = 5.0         # seconds per frame
    T  = 1200        # frames (total time = T*dt)
    n  = 24          # traces

    # Dataset A: fixed 50 s delay
    A1, B1 = synth_dataset_fixed_delay(n_traces=n, T=T, dt=dt,
                                       mean_step_every_s=90.0,
                                       fixed_delay_s=50.0,
                                       step_height=0.08, noise_sd=0.02,
                                       bleach_tau_s=np.inf)
    outA = plot_delay_dashboard(A1, B1, dt=dt,
                                reference="first",
                                smooth_win=7,
                                min_separation_s=60.0,
                                amp_q=0.7,
                                search_window_s=240.0,
                                window_s=180.0)
    print(f"[Dataset A] events={outA['n_events']}, median delay ≈ {outA['median_delay_s']:.1f}s (true=50s)")

    # Dataset B: variable delays (mean~50 s, SD~25 s), drift + dropouts + noise
    A2, B2 = synth_dataset_variable_delay(n_traces=n, T=T, dt=dt,
                                          mean_step_every_s=90.0,
                                          delay_mean_s=50.0, delay_sd_s=25.0,
                                          step_height=0.08, noise_sd=0.03,
                                          missing_prob=0.15,
                                          common_drift_amp=0.15,
                                          drift_period_s=1200.0,
                                          bleach_tau_s=1800.0)
    outB = plot_delay_dashboard(A2, B2, dt=dt,
                                reference="first",
                                smooth_win=7,
                                min_separation_s=60.0,
                                amp_q=0.7,
                                search_window_s=300.0,
                                window_s=200.0)
    print(f"[Dataset B] events={outB['n_events']}, median delay ≈ {outB['median_delay_s']:.1f}s (true~50±25s)")